# PCA vs. no PCA on TENX95, with a spatial-block split (Colab)

**Question:** is ridge on all 4,608 GenBio-PathFM numbers really better than ridge on PCA 256, once the split stops leaking?

**Why this notebook:** in `pca_vs_nopca_colab.ipynb`, no PCA beat PCA 256 by only 0.007 PCC (0.888 vs 0.881), on a random spot split. A random split puts neighbouring spots in train and test, so a model can score well by memorizing its neighbours. A bigger model (4,608 features) can memorize more, so that leak may favour no PCA.

**What is different from the last notebook**

| | Last notebook | This notebook |
|---|---|---|
| Split | random spots, 80/20 | **5 spatial blocks**: hold out one whole region at a time |
| Buffer | none | training spots within **2 spots** of the test block are dropped |
| Choosing λ | leave-one-out on training spots (also leaks) | **inner block CV**: training spots split into 4 regions, λ picked on held-out regions |
| Repeats | 1 split | 5 folds, compared **in pairs** (same fold, both settings) |
| Embeddings | computed on the GPU | **loaded from Drive** (already computed, no 14-min wait) |

Everything that learns from data (gene choice, scaler, PCA, λ) is fit on the training spots of each fold only.

**Decision rule (set before running):** drop PCA only if no PCA beats PCA 256 by **at least 0.01 mean PCC** AND wins in **at least 4 of 5 folds**. Otherwise keep PCA 256.

**Runtime:** a few minutes. A GPU makes the ridge fits faster but is not required.

**What it saves** (in `<your folder>/spatial_blocks/`)

| File | Contents |
|---|---|
| `TENX95_blocks_per_fold.csv` | One row per fold and setting: PCC, NRMSE, RMSE, spot counts, median λ, time |
| `TENX95_blocks_paired.csv` | One row per fold: no PCA minus PCA 256 (PCC and NRMSE), and genes won |
| `TENX95_blocks.png` | Block map, PCC per fold, and the paired differences |

## 1. Setup

In [ ]:
!pip install -q anndata h5py "pandas==2.2.3"

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
import anndata
import torch
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore", message="Mean of empty slice")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Settings

- `N_BLOCKS`: number of regions the slide is cut into (one is held out per fold)
- `N_INNER_BLOCKS`: regions used inside the training spots to pick λ
- `BUFFER_IN_SPOTS`: training spots this many spots or closer to the test block are dropped
- `PCA_SIZES`: PCA settings to compare with no PCA
- `MIN_PCC_GAIN`, `MIN_FOLD_WINS`: the decision rule

In [ ]:
sample_id = "TENX95"
N_GENES = 50
SEED = 0

N_BLOCKS = 5
N_INNER_BLOCKS = 4
BUFFER_IN_SPOTS = 2

PCA_SIZES = [256, 500]
ALPHAS = np.logspace(-2, 5, 30)
MIN_TEST_SD = 0.05

MIN_PCC_GAIN = 0.01
MIN_FOLD_WINS = 4

**Find the folder with the saved embeddings.** The last run saved to `MyDrive/BDD_pca_test`. If your files are somewhere else, add that folder to the list.

In [ ]:
possible_dirs = [
    "/content/drive/MyDrive/BDD_pca_test",
    "/content/drive/MyDrive/BDD_BJMJ/pca_test",
]

DATA_DIR = None
for folder in possible_dirs:
    if os.path.exists(os.path.join(folder, "embeddings_chunks")):
        DATA_DIR = folder
        break

assert DATA_DIR is not None, "No embeddings_chunks folder found. Check where the last notebook saved its files."

OUT_DIR = os.path.join(DATA_DIR, "spatial_blocks")
os.makedirs(OUT_DIR, exist_ok=True)
print("Reading from:", DATA_DIR)
print("Saving to:", OUT_DIR)

## 2. Load embeddings, barcodes, coordinates and expression

In [ ]:
patches_path = os.path.join(DATA_DIR, "hest_data", "patches", f"{sample_id}.h5")
st_path = os.path.join(DATA_DIR, "hest_data", "st", f"{sample_id}.h5ad")

with h5py.File(patches_path, "r") as f:
    raw_barcodes = f["barcode"][:]
    coords = f["coords"][:]

barcodes = []
for b in raw_barcodes:
    if hasattr(b, "ravel"):
        b = b.ravel()[0]
    if isinstance(b, bytes):
        b = b.decode()
    barcodes.append(b)
barcodes = np.array(barcodes)

emb_dir = os.path.join(DATA_DIR, "embeddings_chunks")
chunk_files = sorted(name for name in os.listdir(emb_dir) if name.startswith("chunk_"))
embeddings = np.concatenate([np.load(os.path.join(emb_dir, name)) for name in chunk_files], axis=0)

assert len(embeddings) == len(barcodes), "Embeddings and patches don't line up"
print("Chunks:", len(chunk_files), " embeddings:", embeddings.shape)

adata = anndata.read_h5ad(st_path)
keep = np.isin(barcodes, adata.obs_names)

X_all = embeddings[keep]
coords_kept = coords[keep].astype(float)
counts = adata[barcodes[keep]].X
if hasattr(counts, "toarray"):
    counts = counts.toarray()
log_expr = np.log1p(np.asarray(counts, dtype=np.float32))
all_gene_names = np.array(adata.var_names)

print("X:", X_all.shape, " log1p expression:", log_expr.shape)

## 3. Cut the slide into spatial blocks

KMeans on the spot coordinates gives 5 compact regions. Spot spacing is the median distance to the nearest spot; the buffer is `BUFFER_IN_SPOTS` times that.

In [ ]:
blocks = KMeans(n_clusters=N_BLOCKS, random_state=SEED, n_init=10).fit_predict(coords_kept)

nn = NearestNeighbors(n_neighbors=2).fit(coords_kept)
distances, _ = nn.kneighbors(coords_kept)
spot_spacing = np.median(distances[:, 1])
buffer_distance = BUFFER_IN_SPOTS * spot_spacing * 1.01

print("Spots per block:", np.bincount(blocks))
print(f"Spot spacing: {spot_spacing:.1f} px   buffer: {buffer_distance:.1f} px")

plt.figure(figsize=(5, 5))
plt.scatter(coords_kept[:, 0], coords_kept[:, 1], c=blocks, cmap="tab10", s=3)
plt.gca().invert_yaxis()
plt.gca().set_aspect("equal")
plt.title("Spatial blocks (one is the test set per fold)")
plt.show()

## 4. Helper functions

**Scoring** is the same as the last notebook.

**Ridge for many λ at once:** for one training set, ridge predictions for every λ come from one SVD of the training features. That is much faster than refitting 30 times, and it gives exactly the same answer as sklearn's `Ridge` (with intercept).

**Choosing λ:** the training spots are split into `N_INNER_BLOCKS` regions. For each λ, predict each region from the others and add up the squared errors. Each gene gets the λ with the smallest error.

In [ ]:
def rmse_one_gene(true_values, predicted_values):
    errors = true_values - predicted_values
    return np.sqrt(np.mean(errors ** 2))


def pcc_one_gene(true_values, predicted_values):
    if np.std(true_values) < 1e-6 or np.std(predicted_values) < 1e-6:
        return np.nan
    return np.corrcoef(true_values, predicted_values)[0, 1]


def score_all_genes(y_true, y_pred):
    rmses, nrmses, pccs = [], [], []
    for g in range(y_true.shape[1]):
        true_values = y_true[:, g]
        predicted_values = y_pred[:, g]
        rmse = rmse_one_gene(true_values, predicted_values)
        sd = np.std(true_values)
        rmses.append(rmse)
        if sd < MIN_TEST_SD:
            nrmses.append(np.nan)
            pccs.append(np.nan)
        else:
            nrmses.append(rmse / sd)
            pccs.append(pcc_one_gene(true_values, predicted_values))
    return np.nanmean(rmses), np.nanmean(nrmses), np.nanmean(pccs), np.array(pccs)


def ridge_predictions_all_alphas(X_train, y_train, X_new, alphas):
    # returns shape (number of alphas, number of new spots, number of genes)
    x_mean = X_train.mean(axis=0)
    y_mean = y_train.mean(axis=0)

    X_train_c = torch.tensor(X_train - x_mean, dtype=torch.float32, device=device)
    X_new_c = torch.tensor(X_new - x_mean, dtype=torch.float32, device=device)
    y_train_c = torch.tensor(y_train - y_mean, dtype=torch.float32, device=device)

    U, s, Vt = torch.linalg.svd(X_train_c, full_matrices=False)
    A = X_new_c @ Vt.T
    B = U.T @ y_train_c

    predictions = []
    for alpha in alphas:
        shrink = s / (s ** 2 + alpha)
        pred = (A * shrink) @ B
        predictions.append(pred.cpu().numpy() + y_mean)
    return np.stack(predictions)


def choose_alpha_index_per_gene(X_train, y_train, inner_groups, alphas):
    squared_errors = np.zeros((len(alphas), y_train.shape[1]))
    for group in np.unique(inner_groups):
        is_val = inner_groups == group
        preds = ridge_predictions_all_alphas(X_train[~is_val], y_train[~is_val], X_train[is_val], alphas)
        errors = preds - y_train[is_val]
        squared_errors += (errors ** 2).sum(axis=1)
    return np.argmin(squared_errors, axis=0)


def fit_and_predict(X_train, y_train, X_test, inner_groups):
    best_index = choose_alpha_index_per_gene(X_train, y_train, inner_groups, ALPHAS)
    preds_all = ridge_predictions_all_alphas(X_train, y_train, X_test, ALPHAS)
    y_pred = np.zeros((len(X_test), y_train.shape[1]))
    for g in range(y_train.shape[1]):
        y_pred[:, g] = preds_all[best_index[g], :, g]
    return y_pred, best_index

## 5. Run the 5 folds

For each fold: hold out one block, drop training spots inside the buffer, pick the top 50 genes from the training spots, scale, then run every setting on the **same** spots.

In [ ]:
fold_rows = []
pcc_per_gene = {}

for test_block in range(N_BLOCKS):
    test_idx = np.where(blocks == test_block)[0]
    other_idx = np.where(blocks != test_block)[0]

    nn_test = NearestNeighbors(n_neighbors=1).fit(coords_kept[test_idx])
    dist_to_test, _ = nn_test.kneighbors(coords_kept[other_idx])
    train_idx = other_idx[dist_to_test[:, 0] > buffer_distance]
    n_dropped = len(other_idx) - len(train_idx)

    gene_variance = log_expr[train_idx].var(axis=0)
    top_gene_idx = np.argsort(gene_variance)[::-1][:N_GENES]
    y_train = log_expr[train_idx][:, top_gene_idx]
    y_test = log_expr[test_idx][:, top_gene_idx]

    scaler = StandardScaler()
    Z_train = scaler.fit_transform(X_all[train_idx])
    Z_test = scaler.transform(X_all[test_idx])

    inner_groups = KMeans(n_clusters=N_INNER_BLOCKS, random_state=SEED, n_init=10).fit_predict(coords_kept[train_idx])

    print(f"Fold {test_block}: train {len(train_idx)} (buffer dropped {n_dropped}), test {len(test_idx)}")

    y_pred_mean = np.tile(y_train.mean(axis=0), (len(y_test), 1))
    rmse, nrmse, pcc, _ = score_all_genes(y_test, y_pred_mean)
    fold_rows.append({"fold": test_block, "setting": "mean predictor", "n_train": len(train_idx),
                      "n_test": len(test_idx), "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                      "median_alpha": np.nan, "alphas_at_grid_edge": 0, "seconds": 0})

    settings = []
    for k in PCA_SIZES:
        settings.append((f"PCA {k}", k))
    settings.append(("No PCA", None))

    for name, k in settings:
        t0 = time.time()
        if k is None:
            F_train, F_test = Z_train, Z_test
        else:
            pca = PCA(n_components=k, random_state=SEED)
            F_train = pca.fit_transform(Z_train)
            F_test = pca.transform(Z_test)

        y_pred, best_index = fit_and_predict(F_train, y_train, F_test, inner_groups)
        rmse, nrmse, pcc, pccs = score_all_genes(y_test, y_pred)
        pcc_per_gene[(test_block, name)] = pccs

        at_edge = np.sum((best_index == 0) | (best_index == len(ALPHAS) - 1))
        seconds = time.time() - t0
        fold_rows.append({"fold": test_block, "setting": name, "n_train": len(train_idx),
                          "n_test": len(test_idx), "RMSE": rmse, "NRMSE": nrmse, "PCC": pcc,
                          "median_alpha": float(np.median(ALPHAS[best_index])),
                          "alphas_at_grid_edge": int(at_edge), "seconds": seconds})
        print(f"   {name:>8}: PCC {pcc:.3f} | NRMSE {nrmse:.3f} | {seconds:.0f} s", flush=True)

fold_table = pd.DataFrame(fold_rows)

## 6. Results

**Per-fold table**, then the **paired comparison**: for each fold, no PCA minus PCA 256. Positive PCC difference and negative NRMSE difference mean no PCA did better.

In [ ]:
print(fold_table.round(3).to_string(index=False))

print()
print("Mean over folds:")
print(fold_table.groupby("setting")[["PCC", "NRMSE", "RMSE", "seconds"]].mean().round(3))

In [ ]:
pcc_wide = fold_table.pivot(index="fold", columns="setting", values="PCC")
nrmse_wide = fold_table.pivot(index="fold", columns="setting", values="NRMSE")

paired_rows = []
for fold in range(N_BLOCKS):
    pcc_nopca = pcc_per_gene[(fold, "No PCA")]
    pcc_pca = pcc_per_gene[(fold, "PCA 256")]
    both_valid = ~np.isnan(pcc_nopca) & ~np.isnan(pcc_pca)
    genes_won = int(np.sum(pcc_nopca[both_valid] > pcc_pca[both_valid]))
    paired_rows.append({"fold": fold,
                        "PCC_diff": pcc_wide.loc[fold, "No PCA"] - pcc_wide.loc[fold, "PCA 256"],
                        "NRMSE_diff": nrmse_wide.loc[fold, "No PCA"] - nrmse_wide.loc[fold, "PCA 256"],
                        "genes_won_by_no_pca": genes_won,
                        "genes_compared": int(both_valid.sum())})
paired = pd.DataFrame(paired_rows)
print(paired.round(4).to_string(index=False))

mean_gain = paired["PCC_diff"].mean()
sd_gain = paired["PCC_diff"].std()
fold_wins = int(np.sum(paired["PCC_diff"] > 0))

print()
print(f"No PCA minus PCA 256, PCC: mean {mean_gain:+.4f}, sd {sd_gain:.4f}")
print(f"Folds won by no PCA: {fold_wins} / {N_BLOCKS}")
print()
if mean_gain >= MIN_PCC_GAIN and fold_wins >= MIN_FOLD_WINS:
    print("DECISION: no PCA passes the rule -> dropping PCA is justified on this slide.")
else:
    print("DECISION: no PCA does not pass the rule -> keep PCA 256.")

In [ ]:
fold_table.to_csv(os.path.join(OUT_DIR, f"{sample_id}_blocks_per_fold.csv"), index=False)
paired.to_csv(os.path.join(OUT_DIR, f"{sample_id}_blocks_paired.csv"), index=False)
print("Saved CSVs to", OUT_DIR)

In [ ]:
model_settings = [f"PCA {k}" for k in PCA_SIZES] + ["No PCA"]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].scatter(coords_kept[:, 0], coords_kept[:, 1], c=blocks, cmap="tab10", s=3)
axes[0].invert_yaxis()
axes[0].set_aspect("equal")
axes[0].set_title("Spatial blocks")
axes[0].set_xticks([])
axes[0].set_yticks([])

x_positions = np.arange(len(model_settings))
for fold in range(N_BLOCKS):
    axes[1].plot(x_positions, [pcc_wide.loc[fold, s] for s in model_settings], marker="o", label=f"block {fold}")
axes[1].set_xticks(x_positions)
axes[1].set_xticklabels(model_settings)
axes[1].set_ylabel("Mean PCC (test block)")
axes[1].set_title("PCC per held-out block")
axes[1].legend(fontsize=8)

axes[2].bar(paired["fold"], paired["PCC_diff"])
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].axhline(MIN_PCC_GAIN, color="red", linestyle="--", label=f"decision threshold (+{MIN_PCC_GAIN})")
axes[2].set_xlabel("Held-out block")
axes[2].set_ylabel("PCC: No PCA minus PCA 256")
axes[2].set_title("Paired difference (above 0 = no PCA better)")
axes[2].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"{sample_id}_blocks.png"), dpi=150, bbox_inches="tight")
plt.show()

### How to read it

- **Compare with the random split.** There, no PCA beat PCA 256 by +0.007 PCC. If the mean difference here shrinks or turns negative, that gain was mostly no PCA memorizing neighbouring spots.
- **Look at the folds, not just the mean.** PCC varies a lot between blocks (different tissue in each region), which is why the comparison is paired: both settings are scored on the same block.
- **Genes won** shows whether a win is broad (most of the 50 genes) or driven by a few genes.
- **`alphas_at_grid_edge`** should be 0 or close to it. If many genes pick the smallest or largest λ, widen `ALPHAS`.
- **Absolute PCC will be lower** than in the random split. That is expected and is closer to the true performance.
- **Still one slide, one patient.** If no PCA passes the rule here, confirm with leave-one-patient-out across slides before switching.